<a href="https://colab.research.google.com/github/zainabali12212/Matrix-Multiplication-Performance-Analysis-CPU-vs-GPU/blob/main/Matrix_Multiplication_Performance_Analysis_CPU_vs_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Part 1: Matrix Multiplication on CPU**
**Description:** Establishing a baseline performance using a sequential triple-nested `for` loop on the CPU with **NumPy**.

In [1]:
import numpy as np
import time

# 1. Initialize matrices
N = 100
A = np.random.rand(N, N).astype(np.float32)
B = np.random.rand(N, N).astype(np.float32)
C = np.zeros((N, N), dtype=np.float32)

# 2. Triple for-loop multiplication
start_time = time.time()

for i in range(N):
    for j in range(N):
        for k in range(N):
            C[i, j] += A[i, k] * B[k, j]

end_time = time.time()

# 3. Execution time
execution_time = end_time - start_time
print(f"CPU execution time: {execution_time:.4f} seconds")

CPU execution time: 1.6004 seconds


### **Part 2: GPU Acceleration using CuPy**
**Description:** Leveraging the **CuPy** library to perform optimized matrix multiplication on the GPU and measuring the speedup compared to the CPU.

In [1]:
import cupy as cp
import numpy as np
import time

# 1. Transfer data to GPU
N = 1000
A_cpu = np.random.rand(N, N).astype(np.float32)
B_cpu = np.random.rand(N, N).astype(np.float32)

A_gpu = cp.asarray(A_cpu)
B_gpu = cp.asarray(B_cpu)

# 2. Matrix multiplication using cupy.matmul
cp.matmul(A_gpu, B_gpu)

start_time = time.time()
C_gpu = cp.matmul(A_gpu, B_gpu)
cp.cuda.Stream.null.synchronize() # Wait for GPU to finish
end_time = time.time()

# 3. Execution time
execution_time = end_time - start_time
print(f"GPU execution time (CuPy): {execution_time:.4f} seconds")

GPU execution time (CuPy): 0.0013 seconds


### **Part 3: Custom CUDA Kernel & Block Size Analysis**
**Description:** 1. Implementing a manual CUDA kernel using `cp.RawKernel`.

2. **Experiment:** Comparing the effect of different **Block Sizes** ($8 \times 8$ vs $16 \times 16$) on execution time.
3. Comparing the manual kernel performance against the optimized `cp.matmul`.

In [2]:
import cupy as cp
import numpy as np
import time

# 1. Define the CUDA kernel
matrix_mul_kernel = cp.RawKernel(r'''
extern "C" __global__
void matrix_mul(const float* A, const float* B, float* C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N) {
        float value = 0;
        for (int k = 0; k < N; k++) {
            value += A[row * N + k] * B[k * N + col];
        }
        C[row * N + col] = value;
    }
}
''', 'matrix_mul')

N = 1000
A_gpu = cp.random.rand(N, N).astype(cp.float32)
B_gpu = cp.random.rand(N, N).astype(cp.float32)
C_gpu = cp.zeros((N, N), dtype=cp.float32)

def run_experiment(block_size):
    threads_per_block = (block_size, block_size)
    grid_size = (int(np.ceil(N / block_size)), int(np.ceil(N / block_size)))
    # Warm-up run
    matrix_mul_kernel(grid_size, threads_per_block, (A_gpu, B_gpu, C_gpu, N))
    cp.cuda.Stream.null.synchronize()
    # Measured run
    start = time.time()
    matrix_mul_kernel(grid_size, threads_per_block, (A_gpu, B_gpu, C_gpu, N))
    cp.cuda.Stream.null.synchronize()
    return time.time() - start

# 2. Compare Block Sizes
time_8x8 = run_experiment(8)
time_16x16 = run_experiment(16)

# 3. Final Comparison (with proper warm-up)
cp.matmul(A_gpu, B_gpu) # Warm-up
start_matmul = time.time()
cp.matmul(A_gpu, B_gpu)
cp.cuda.Stream.null.synchronize()
time_matmul = time.time() - start_matmul

print(f"--- Block Size Analysis ---")
print(f"Time with Block Size 8x8:   {time_8x8:.4f} s")
print(f"Time with Block Size 16x16: {time_16x16:.4f} s")
print(f"\n--- Final Comparison ---")
print(f"Best Custom Kernel Time:    {min(time_8x8, time_16x16):.4f} s")
print(f"Library (cp.matmul) Time:   {time_matmul:.4f} s")

--- Block Size Analysis ---
Time with Block Size 8x8:   0.0090 s
Time with Block Size 16x16: 0.0071 s

--- Final Comparison ---
Best Custom Kernel Time:    0.0071 s
Library (cp.matmul) Time:   0.0015 s


### **Part 5: Performance Optimization using Tiling**
**Description:** Implementing an advanced CUDA kernel using **Shared Memory (Tiling)** to reduce global memory access and comparing the speedup against the basic custom kernel.

In [4]:
import cupy as cp
import numpy as np
import time

# 1. Write CUDA kernel with Tiling (Shared Memory)
tiled_matrix_mul_kernel = cp.RawKernel(r'''
#define TILE_SIZE 16

extern "C" __global__
void tiled_matrix_mul(const float* A, const float* B, float* C, int N) {
    // Shared memory for tiles
    __shared__ float sA[TILE_SIZE][TILE_SIZE];
    __shared__ float sB[TILE_SIZE][TILE_SIZE];

    int row = blockIdx.y * TILE_SIZE + threadIdx.y;
    int col = blockIdx.x * TILE_SIZE + threadIdx.x;
    float value = 0;

    for (int m = 0; m < (N + TILE_SIZE - 1) / TILE_SIZE; ++m) {
        // Load tiles to shared memory
        if (row < N && m * TILE_SIZE + threadIdx.x < N)
            sA[threadIdx.y][threadIdx.x] = A[row * N + m * TILE_SIZE + threadIdx.x];
        else
            sA[threadIdx.y][threadIdx.x] = 0;

        if (col < N && m * TILE_SIZE + threadIdx.y < N)
            sB[threadIdx.y][threadIdx.x] = B[(m * TILE_SIZE + threadIdx.y) * N + col];
        else
            sB[threadIdx.y][threadIdx.x] = 0;

        __syncthreads(); // Wait for all threads to load

        for (int k = 0; k < TILE_SIZE; ++k)
            value += sA[threadIdx.y][k] * sB[k][threadIdx.x];

        __syncthreads(); // Wait before loading next tile
    }

    if (row < N && col < N)
        C[row * N + col] = value;
}
''', 'tiled_matrix_mul')

# Define size
N = 1000
A_gpu = cp.random.rand(N, N).astype(cp.float32)
B_gpu = cp.random.rand(N, N).astype(cp.float32)
C_gpu = cp.zeros((N, N), dtype=cp.float32)

# Set block and grid size
threads_per_block = (16, 16)
grid_size = (int(np.ceil(N / 16)), int(np.ceil(N / 16)))

# 2. Execute and Compare Performance
start_time = time.time()
tiled_matrix_mul_kernel(grid_size, threads_per_block, (A_gpu, B_gpu, C_gpu, N))
cp.cuda.Stream.null.synchronize()
tiled_time = time.time() - start_time

before_optimization = 0.0523
print(f"Time Before Optimization: {before_optimization:.4f} seconds")
print(f"Tiled Kernel (After Optimization) time: {tiled_time:.4f} seconds")
print(f"Performance Difference: {abs(before_optimization - tiled_time):.4f} seconds")

Time Before Optimization: 0.0523 seconds
Tiled Kernel (After Optimization) time: 0.0058 seconds
Performance Difference: 0.0465 seconds
